<a id="encrypted-finance-tutorial-transactions"></a>
# Encrypted Finance Tutorial: Transactions

This tutorial covers `src/concrete_fhe_toolkit/finance/transactions.py`. In FHE, conditional branches (`if`) are not allowed because they leak information about the data. The `transfer` function demonstrates how to securely process a money transfer where the sender's balance is only deducted if it's sufficient, without ever revealing if the transfer succeeded or failed to the server.

<a id="secure-money-transfer"></a>
## Secure Money Transfer

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.finance.transactions import transfer

def test_transfer(sender_bal: int, receiver_bal: int, amount: int):
    new_sender, new_receiver = transfer(sender_bal, receiver_bal, amount)
    return new_sender, new_receiver

compiler = fhe.Compiler(test_transfer, {"sender_bal": "encrypted", "receiver_bal": "encrypted", "amount": "encrypted"})
inputset = [(1000, 500, 200), (100, 500, 200)]
circuit = compiler.compile(inputset)

# Case 1: Sender has enough money (1000 > 200)
res_success = circuit.encrypt_run_decrypt(1000, 500, 200)
assert res_success == (800, 700) # Sender: 1000-200, Receiver: 500+200
print("✅ Successful transfer passed!")

# Case 2: Sender DOES NOT have enough money (100 < 200)
res_fail = circuit.encrypt_run_decrypt(100, 500, 200)
assert res_fail == (100, 500) # Transfer is silently cancelled
print("✅ Failed transfer (silently cancelled) passed!")